# Phân tích kết quả Midterm ASR

Notebook này đọc file `outputs/model_comparison_6metrics.csv`, làm sạch dữ liệu, giải thích 6 metric và vẽ biểu đồ trực quan cho 3 mô hình:

- `Whisper-base zero-shot`
- `PhoWhisper-base zero-shot`
- `PhoWhisper tone-aware LoRA`

Các giá trị đều là error rate, nên **càng thấp càng tốt**.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Markdown, display

RESULTS_PATH = Path("../outputs/model_comparison_6metrics.csv")
if not RESULTS_PATH.exists():
    RESULTS_PATH = Path("outputs/model_comparison_6metrics.csv")

metrics = ["wer", "cer", "ter_simple", "der_simple", "fcer_simple", "swdr_simple"]
metric_names = {
    "wer": "WER",
    "cer": "CER",
    "ter_simple": "TER",
    "der_simple": "DER",
    "fcer_simple": "FCER",
    "swdr_simple": "SWDR",
}
model_order = [
    "Whisper-base zero-shot",
    "PhoWhisper-base zero-shot",
    "PhoWhisper tone-aware LoRA",
]
condition_order = ["clean", "noisy_all", "snr_20", "snr_10", "snr_5", "snr_0"]

df = pd.read_csv(RESULTS_PATH, skipinitialspace=True)
df.columns = [column.strip() for column in df.columns]
for column in ["model", "condition"]:
    df[column] = df[column].astype(str).str.strip()
for column in ["n", *metrics]:
    df[column] = pd.to_numeric(df[column].astype(str).str.strip(), errors="coerce")

df["model"] = pd.Categorical(df["model"], categories=model_order, ordered=True)
df["condition"] = pd.Categorical(df["condition"], categories=condition_order, ordered=True)
df = df.sort_values(["model", "condition"]).reset_index(drop=True)

print(f"Loaded: {RESULTS_PATH}")
print(f"Rows: {len(df)}")
display(df)

Loaded: ..\outputs\model_comparison_6metrics.csv
Rows: 18


,model,condition,n,wer,cer,ter_simple,der_simple,fcer_simple,swdr_simple
0,Whisper-base zero-shot,clean,30,0.430605,0.196078,0.231317,0.122222,0.227848,0.043478
1,Whisper-base zero-shot,noisy_all,120,0.480427,0.236928,0.260676,0.105590,0.305380,0.032609
2,Whisper-base zero-shot,snr_20,30,0.434164,0.200980,0.209964,0.101695,0.240506,0.000000
3,Whisper-base zero-shot,snr_10,30,0.498221,0.242647,0.252669,0.090323,0.322785,0.043478
4,Whisper-base zero-shot,snr_5,30,0.459075,0.225490,0.259786,0.107784,0.303797,0.043478
5,Whisper-base zero-shot,snr_0,30,0.530249,0.278595,0.320285,0.124138,0.354430,0.043478
6,PhoWhisper-base zero-shot,clean,30,0.081851,0.046569,0.024911,0.003937,0.044304,0.000000
7,PhoWhisper-base zero-shot,noisy_all,120,0.118327,0.064134,0.032918,0.007107,0.060127,0.000000
8,PhoWhisper-base zero-shot,snr_20,30,0.088968,0.048203,0.028470,0.011811,0.044304,0.000000
9,PhoWhisper-base zero-shot,snr_10,30,0.092527,0.054739,0.028470,0.003968,0.050633,0.000000


## Ý nghĩa 6 metric

- **WER**: Word Error Rate, lỗi ở cấp từ.
- **CER**: Character Error Rate, lỗi ở cấp ký tự.
- **TER**: Tone Error Rate bản đơn giản, lỗi thanh điệu trong tiếng Việt.
- **DER**: Diacritic Error Rate bản đơn giản, lỗi dấu/diacritic khi phần chữ nền giống nhau.
- **FCER**: Final Consonant Error Rate bản đơn giản, lỗi phụ âm cuối như `c/ch/m/n/ng/nh/p/t`.
- **SWDR**: Short Word Deletion Rate bản đơn giản, tỷ lệ từ ngắn bị xóa mất.

TER/DER/FCER/SWDR đang là diagnostic metrics cho midterm, nên nên đọc như chỉ báo lỗi theo kiểu tiếng Việt, không phải benchmark chuẩn cuối cùng.

In [2]:
def percent(value: float) -> str:
    return f"{value * 100:.2f}%"


display_df = df.copy()
for column in metrics:
    display_df[column] = display_df[column].map(percent)

display(display_df)

,model,condition,n,wer,cer,ter_simple,der_simple,fcer_simple,swdr_simple
0,Whisper-base zero-shot,clean,30,43.06%,19.61%,23.13%,12.22%,22.78%,4.35%
1,Whisper-base zero-shot,noisy_all,120,48.04%,23.69%,26.07%,10.56%,30.54%,3.26%
2,Whisper-base zero-shot,snr_20,30,43.42%,20.10%,21.00%,10.17%,24.05%,0.00%
3,Whisper-base zero-shot,snr_10,30,49.82%,24.26%,25.27%,9.03%,32.28%,4.35%
4,Whisper-base zero-shot,snr_5,30,45.91%,22.55%,25.98%,10.78%,30.38%,4.35%
5,Whisper-base zero-shot,snr_0,30,53.02%,27.86%,32.03%,12.41%,35.44%,4.35%
6,PhoWhisper-base zero-shot,clean,30,8.19%,4.66%,2.49%,0.39%,4.43%,0.00%
7,PhoWhisper-base zero-shot,noisy_all,120,11.83%,6.41%,3.29%,0.71%,6.01%,0.00%
8,PhoWhisper-base zero-shot,snr_20,30,8.90%,4.82%,2.85%,1.18%,4.43%,0.00%
9,PhoWhisper-base zero-shot,snr_10,30,9.25%,5.47%,2.85%,0.40%,5.06%,0.00%


In [3]:
colors = {
    "Whisper-base zero-shot": "#64748B",
    "PhoWhisper-base zero-shot": "#2563EB",
    "PhoWhisper tone-aware LoRA": "#16A34A",
}


def display_grouped_bars(data: pd.DataFrame, metric: str, title: str, height: int = 420) -> None:
    data = data.dropna(subset=[metric]).copy()
    conditions = [item for item in condition_order if item in set(data["condition"].astype(str))]
    models = [item for item in model_order if item in set(data["model"].astype(str))]
    max_value = max(float(data[metric].max()) * 1.18, 0.01)

    width = 1040
    margin_left = 80
    margin_right = 30
    margin_top = 58
    margin_bottom = 96
    chart_width = width - margin_left - margin_right
    chart_height = height - margin_top - margin_bottom
    group_width = chart_width / max(len(conditions), 1)
    bar_width = min(42, group_width / max(len(models) + 1, 1))

    svg = [
        f'<svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg">',
        '<rect width="100%" height="100%" fill="#FFFFFF"/>',
        f'<text x="{margin_left}" y="30" font-size="22" font-weight="700" fill="#0F172A">{title}</text>',
    ]

    for tick in range(6):
        value = max_value * tick / 5
        y = margin_top + chart_height - (value / max_value) * chart_height
        svg.append(f'<line x1="{margin_left}" x2="{width - margin_right}" y1="{y:.1f}" y2="{y:.1f}" stroke="#E2E8F0"/>')
        svg.append(f'<text x="{margin_left - 12}" y="{y + 4:.1f}" text-anchor="end" font-size="12" fill="#475569">{value * 100:.0f}%</text>')

    for group_index, condition in enumerate(conditions):
        group_x = margin_left + group_index * group_width
        label_x = group_x + group_width / 2
        svg.append(f'<text x="{label_x:.1f}" y="{height - 52}" text-anchor="middle" font-size="13" fill="#334155">{condition}</text>')
        for model_index, model in enumerate(models):
            row = data[(data["condition"].astype(str) == condition) & (data["model"].astype(str) == model)]
            if row.empty:
                continue
            value = float(row.iloc[0][metric])
            bar_height = (value / max_value) * chart_height
            x = group_x + (group_width - bar_width * len(models)) / 2 + model_index * bar_width
            y = margin_top + chart_height - bar_height
            svg.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_width - 4:.1f}" height="{bar_height:.1f}" rx="4" fill="{colors[model]}"/>')
            svg.append(f'<text x="{x + (bar_width - 4) / 2:.1f}" y="{y - 6:.1f}" text-anchor="middle" font-size="11" fill="#0F172A">{value * 100:.1f}%</text>')

    legend_x = margin_left
    legend_y = height - 24
    for model in models:
        svg.append(f'<rect x="{legend_x}" y="{legend_y - 12}" width="14" height="14" rx="3" fill="{colors[model]}"/>')
        svg.append(f'<text x="{legend_x + 20}" y="{legend_y}" font-size="13" fill="#334155">{model}</text>')
        legend_x += 270

    svg.append('</svg>')
    display(HTML("".join(svg)))


def display_metric_heatmap(data: pd.DataFrame, condition: str) -> None:
    subset = data[data["condition"].astype(str) == condition].copy()
    rows = []
    for _, row in subset.iterrows():
        cells = [f'<td style="font-weight:700; padding:8px 10px; white-space:nowrap;">{row["model"]}</td>']
        for metric in metrics:
            value = float(row[metric])
            max_value = max(float(subset[metric].max()), 0.001)
            alpha = 0.12 + 0.58 * (value / max_value)
            cells.append(
                f'<td style="padding:8px 10px; text-align:right; background:rgba(239,68,68,{alpha:.2f});">{value * 100:.2f}%</td>'
            )
        rows.append(f'<tr>{"".join(cells)}</tr>')

    headers = ''.join([f'<th style="padding:8px 10px; text-align:right;">{metric_names[m]}</th>' for m in metrics])
    html = f'''
    <h3 style="font-family:Segoe UI, sans-serif; margin-bottom:8px;">Heatmap 6 metrics - {condition}</h3>
    <table style="border-collapse:collapse; font-family:Segoe UI, sans-serif; font-size:14px; border:1px solid #CBD5E1;">
      <thead><tr><th style="padding:8px 10px; text-align:left;">Model</th>{headers}</tr></thead>
      <tbody>{''.join(rows)}</tbody>
    </table>
    <p style="font-family:Segoe UI, sans-serif; color:#475569;">Màu đỏ đậm hơn nghĩa là error rate cao hơn trong cùng metric.</p>
    '''
    display(HTML(html))

## Biểu đồ tổng quan

Các biểu đồ dưới đây so sánh trực tiếp 3 mô hình. Vì tất cả là error rate, cột thấp hơn là tốt hơn.

In [4]:
overview = df[df["condition"].astype(str).isin(["clean", "noisy_all"])]
snr_rows = df[df["condition"].astype(str).str.startswith("snr_")]

display_grouped_bars(overview, "wer", "WER: Clean vs Noisy all")
display_grouped_bars(overview, "cer", "CER: Clean vs Noisy all")
display_grouped_bars(snr_rows, "wer", "WER theo SNR")
display_grouped_bars(snr_rows, "fcer_simple", "FCER theo SNR")

## Heatmap 6 metric

Heatmap giúp nhìn nhanh mô hình nào đang có lỗi cao hơn theo từng loại metric.

In [5]:
display_metric_heatmap(df, "clean")
display_metric_heatmap(df, "noisy_all")

Model,WER,CER,TER,DER,FCER,SWDR
Whisper-base zero-shot,43.06%,19.61%,23.13%,12.22%,22.78%,4.35%
PhoWhisper-base zero-shot,8.19%,4.66%,2.49%,0.39%,4.43%,0.00%
PhoWhisper tone-aware LoRA,9.25%,4.49%,1.42%,0.00%,5.06%,0.00%


Model,WER,CER,TER,DER,FCER,SWDR
Whisper-base zero-shot,48.04%,23.69%,26.07%,10.56%,30.54%,3.26%
PhoWhisper-base zero-shot,11.83%,6.41%,3.29%,0.71%,6.01%,0.00%
PhoWhisper tone-aware LoRA,11.39%,5.82%,2.76%,0.70%,6.01%,0.00%


## Tóm tắt tự động

Cell này tạo bullet summary từ số liệu hiện tại để dùng trong slide/report.

In [6]:
def get_value(model: str, condition: str, metric: str) -> float:
    row = df[(df["model"].astype(str) == model) & (df["condition"].astype(str) == condition)]
    if row.empty:
        raise ValueError(f"Missing row: {model} / {condition}")
    return float(row.iloc[0][metric])


whisper_noisy_wer = get_value("Whisper-base zero-shot", "noisy_all", "wer")
phowhisper_noisy_wer = get_value("PhoWhisper-base zero-shot", "noisy_all", "wer")
lora_noisy_wer = get_value("PhoWhisper tone-aware LoRA", "noisy_all", "wer")
phowhisper_clean_wer = get_value("PhoWhisper-base zero-shot", "clean", "wer")
lora_clean_wer = get_value("PhoWhisper tone-aware LoRA", "clean", "wer")
lora_noisy_ter = get_value("PhoWhisper tone-aware LoRA", "noisy_all", "ter_simple")
base_noisy_ter = get_value("PhoWhisper-base zero-shot", "noisy_all", "ter_simple")

summary = f"""
- Whisper noisy WER: **{percent(whisper_noisy_wer)}**.
- PhoWhisper zero-shot noisy WER: **{percent(phowhisper_noisy_wer)}**, thấp hơn Whisper rất nhiều trên cùng noisy subset.
- PhoWhisper tone-aware LoRA noisy WER: **{percent(lora_noisy_wer)}**.
- Clean WER của PhoWhisper zero-shot là **{percent(phowhisper_clean_wer)}**, còn LoRA là **{percent(lora_clean_wer)}**.
- TER noisy của PhoWhisper zero-shot là **{percent(base_noisy_ter)}**, còn LoRA là **{percent(lora_noisy_ter)}**.
- Nếu `SWDR = 0.0`, nghĩa là subset đó không phát hiện short-word deletion theo metric hiện tại, không phải lỗi dữ liệu.
"""
display(Markdown(summary))


- Whisper noisy WER: **48.04%**.
- PhoWhisper zero-shot noisy WER: **11.83%**, thấp hơn Whisper rất nhiều trên cùng noisy subset.
- PhoWhisper tone-aware LoRA noisy WER: **11.39%**.
- Clean WER của PhoWhisper zero-shot là **8.19%**, còn LoRA là **9.25%**.
- TER noisy của PhoWhisper zero-shot là **3.29%**, còn LoRA là **2.76%**.
- Nếu `SWDR = 0.0`, nghĩa là subset đó không phát hiện short-word deletion theo metric hiện tại, không phải lỗi dữ liệu.


## Ghi chú khi đưa vào báo cáo

- Dữ liệu noisy được tạo bằng cách mix **VIVOS speech** với **MUSAN noise** theo SNR `20/10/5/0`.
- Whisper và PhoWhisper được so sánh công bằng vì dùng cùng manifest, cùng limit và cùng scoring script.
- `fcer_simple` và `swdr_simple` là metric mới được thêm cho midterm, nên nên gọi là bản simple/prototype.
- Một số giá trị `0.0` là hợp lệ: nghĩa là trong subset đó không phát hiện lỗi loại đó.